# 🚦 3-Kaynaklı Dengeli Trafik Araç Tespiti: UA-DETRAC + MIO-TCD + COCO 2017 (YOLO11s)
### Faz 1 — Merkezi (Centralized) Yüksek Başarımlı Baseline & FL Hazırlığı

Bu notebook 3 tamamlayıcı trafik kaynağını ortak 4-sınıf şemasında (`car`, `motorcycle`, `bus`, `truck`) birleştirir:
1. **UA-DETRAC** (Çin otoyol CCTV kameraları — 4 hava koşulu, `<ignored_region>` maskeleme + adaptive örnekleme)
2. **MIO-TCD Localization** (Kuzey Amerika sabit kavşak/otoyol CCTV kameraları — yüksek `motorcycle` ve ağır vasıta `truck` hacmi)
3. **COCO 2017 Alt Kümesi** (yalnızca `motorcycle`, `truck`, `bus` içeren ~2.300 sokak görseli — 241 MB etiket + seçili görseller)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/fed-traffic-vision/blob/main/notebooks/01_centralized_3source_yolo11s.ipynb)


## Adım 1: T4 GPU Kontrolü ve Bağımlılık Kurulumu


In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU aktif: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Runtime -> Change runtime type -> T4 GPU seçin.')

!pip install -q ultralytics pyyaml tqdm opencv-python-headless
print('✅ Kütüphaneler hazır!')


## Adım 2: GitHub Deposunu Çekme ve Drive Veri Kaynaklarını Doğrulama


In [ ]:
import os
from google.colab import drive

repo_dir = '/content/fed-traffic-vision'
if not os.path.exists(repo_dir):
    !git clone https://github.com/aydin-eng/fed-traffic-vision.git /content/fed-traffic-vision
%cd /content/fed-traffic-vision
!git pull

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('✅ Google Drive zaten bağlı.')

DETRAC_ZIP = '/content/drive/MyDrive/DETRAC-Images.zip'
MIOTCD_TAR = '/content/drive/MyDrive/Datasets/MIO-TCD-Localization.tar'

assert os.path.exists(DETRAC_ZIP), f'Bulunamadı: {DETRAC_ZIP}'
assert os.path.exists(MIOTCD_TAR), f'Bulunamadı: {MIOTCD_TAR}'
print(f'✅ UA-DETRAC: {DETRAC_ZIP} ({os.path.getsize(DETRAC_ZIP)/(1024**3):.2f} GB)')
print(f'✅ MIO-TCD:   {MIOTCD_TAR} ({os.path.getsize(MIOTCD_TAR)/(1024**3):.2f} GB)')


## Adım 3: 3 Kaynaklı Birleşik Veri Setini Oluşturma (`UA-DETRAC + MIO-TCD + COCO 2017`)


In [ ]:
import subprocess

DATASET_DIR = '/content/unified_traffic_dataset'

cmd = [
    'python', 'src/data/build_unified_dataset.py',
    '--out-dir', DATASET_DIR,
    '--detrac-annot-zip', 'detrac_xml_annotations.zip',
    '--detrac-images-zip', DETRAC_ZIP,
    '--miotcd-tar', MIOTCD_TAR,
    '--miotcd-max-train', '4000',
    '--miotcd-max-val', '600',
    '--coco-cache-dir', '/content/coco_cache',
    '--coco-max-train', '2000',
    '--coco-max-val', '300',
    '--rare-copies', '1',
]
print('🚀 3 kaynaklı birleşik veri seti hazırlanıyor...')
subprocess.run(cmd, check=True)


## Adım 4: Kaynak & Sınıf Dengesizliği Raporunu Doğrulama


In [ ]:
import csv
from collections import Counter

with open('/content/unified_traffic_dataset/dataset_metadata.csv', 'r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Toplam kayıtlı kare: {len(rows):,}')
for sp in ('train', 'val', 'test'):
    sp_rows = [r for r in rows if r['split'] == sp]
    src_c = Counter(r['source'] for r in sp_rows)
    boxes = {c: sum(int(r[f'num_{c}']) for r in sp_rows) for c in ('car', 'motorcycle', 'bus', 'truck')}
    print(f'[{sp.upper():<5}] imgs={len(sp_rows):>6,} | kaynaklar={dict(src_c)} | kutular={boxes}')


## Adım 5: YOLO11s Eğitimi (`imgsz=800`, 50 Epoch, `cls=0.8`)


In [ ]:
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo11s.pt')

results = model.train(
    data='/content/unified_traffic_dataset/data.yaml',
    epochs=50,
    imgsz=800,
    batch=16,          # VRAM yetersiz kalırsa batch=8 yapın
    device=device_arg,
    cos_lr=True,
    lr0=0.005,
    cls=0.8,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    close_mosaic=10,
    patience=15,
    save=True,
    plots=True,
    seed=42,
    project='runs/train',
    name='yolo11s_3source_exp',
    verbose=True,
)


## Adım 6: Sınıf Bazlı & Alan Bazlı (Cross-Domain) Değerlendirme


In [ ]:
import glob
import json
import os
from ultralytics import YOLO

exp_dir = 'runs/train/yolo11s_3source_exp'
if not os.path.exists(exp_dir):
    cands = sorted(glob.glob('runs/**/yolo11s_3source_exp*', recursive=True))
    if cands:
        exp_dir = cands[-1]

best_weights = os.path.join(exp_dir, 'weights', 'best.pt')
trained_model = YOLO(best_weights)
metrics = trained_model.val(data='/content/unified_traffic_dataset/data.yaml', imgsz=800)

class_names = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
per_class = {}
print(f'{"Sınıf":<14} | {"Precision":>10} | {"Recall":>10} | {"mAP@50":>10} | {"mAP@50-95":>10}')
print('-' * 66)
for idx, cid in enumerate(metrics.box.ap_class_index):
    cname = class_names.get(int(cid), str(cid))
    p, r = float(metrics.box.p[idx]), float(metrics.box.r[idx])
    m50, m95 = float(metrics.box.ap50[idx]), float(metrics.box.ap[idx])
    per_class[cname] = {'precision': p, 'recall': r, 'mAP50': m50, 'mAP50_95': m95}
    print(f'{cname:<14} | {p:>10.4f} | {r:>10.4f} | {m50:>10.4f} | {m95:>10.4f}')
print('-' * 66)
print(f'{"ALL (Mean)":<14} | {float(metrics.box.mp):>10.4f} | {float(metrics.box.mr):>10.4f} | {float(metrics.box.map50):>10.4f} | {float(metrics.box.map):>10.4f}')

payload = {
    'overall': {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
    },
    'per_class': per_class,
}
with open(os.path.join(exp_dir, 'per_class_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)


## Adım 7: Sonuçları Paketleme ve Google Drive'a Yedekleme 📦


In [ ]:
import glob
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11s_3source_exp'
if not os.path.exists(exp_dir):
    cands = sorted(glob.glob('runs/**/yolo11s_3source_exp*', recursive=True))
    if cands:
        exp_dir = cands[-1]

meta_src = '/content/unified_traffic_dataset/dataset_metadata.csv'
if os.path.exists(meta_src):
    shutil.copy2(meta_src, os.path.join(exp_dir, 'dataset_metadata.csv'))

zip_path = shutil.make_archive('fed_traffic_v1_results', 'zip', exp_dir)
print(f'📦 Paket hazır: {zip_path} ({os.path.getsize(zip_path) / (1024*1024):.1f} MB)')

drive_out_dir = '/content/drive/MyDrive/Colab Notebooks'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_out_dir, exist_ok=True)
    drive_dst = os.path.join(drive_out_dir, os.path.basename(zip_path))
    shutil.copy2(zip_path, drive_dst)
    print(f'☁️ Google Drive yedeği kaydedildi: {drive_dst}')

try:
    files.download(zip_path)
except Exception as e:
    print(f'ℹ️ VS Code Colab çekirdeğinde tarayıcı indirmesi atlandı ({e}).')
    print('👉 Sonuç paketi Drive üzerinde: Colab Notebooks/fed_traffic_v1_results.zip')
